### Pretrained Transformer Model on Sarcasm Detection

In [47]:
from transformers import pipeline

sarcasm_model = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-irony"
)

Device set to use cuda:0


In [48]:
import pandas as pd
import polars as pl

all = pl.read_excel(r'D:\Github\4021-Information-Retrieval\classification\evaluation\evaluation_dataset.xlsx').to_pandas()

In [49]:
def get_sarcasm(text):
    if text is None or str(text).strip() == "":
        return {"label": None, "score": None}
    
    result = sarcasm_model(text[:512])[0]  # truncate long reviews
    return result

In [50]:
sarcasm_results = all["content"].apply(get_sarcasm)

In [51]:
sarcasm_results

0        {'label': 'irony', 'score': 0.8711602687835693}
1      {'label': 'non_irony', 'score': 0.942295730113...
2      {'label': 'non_irony', 'score': 0.816238105297...
3      {'label': 'non_irony', 'score': 0.979485750198...
4      {'label': 'non_irony', 'score': 0.676999330520...
                             ...                        
995    {'label': 'non_irony', 'score': 0.876514017581...
996    {'label': 'non_irony', 'score': 0.820380032062...
997    {'label': 'non_irony', 'score': 0.702274143695...
998    {'label': 'non_irony', 'score': 0.958595395088...
999    {'label': 'non_irony', 'score': 0.643952548503...
Name: content, Length: 1000, dtype: object

In [52]:
all["sarcasm_label"] = sarcasm_results.apply(lambda x: x["label"])
all["sarcasm_score"] = sarcasm_results.apply(lambda x: x["score"])

In [53]:
pl.from_pandas(all).write_csv('pretrained_model_sarcasm.csv')

### Rule Based Sarcasm Detection

In [31]:
import re
import pandas as pd


# =========================
# 1. Define rule resources
# =========================

SARCASTIC_PHRASES = [
    r"\byeah right\b",
    r"\bas if\b",
    r"\boh great\b",
    r"\bhow nice\b",
    r"\bwhat a surprise\b",
    r"\bjust what i needed\b",
    r"\blove that for me\b",
    r"\bgood for you\b",
    r"\bthanks a lot\b",
    r"\bwhat could possibly go wrong\b",
    r"\bso much for\b",
    r"\bof course\b",
    r"\bbrilliant\b",
    r"\bgenius\b",
    r"\bwow\b"
]

POSITIVE_WORDS = {
    "great", "amazing", "awesome", "fantastic", "wonderful", "perfect",
    "excellent", "brilliant", "love", "loved", "best", "incredible",
    "masterpiece", "outstanding", "beautiful", "superb"
}

NEGATIVE_WORDS = {
    "bad", "awful", "terrible", "horrible", "worst", "boring", "dumb",
    "stupid", "waste", "mess", "disappointing", "disappointed", "annoying",
    "ridiculous", "pathetic", "trash", "sucks", "sucked", "hate", "hated",
    "mediocre", "nonsense", "painful", "cringe", "cringey"
}

CONTRAST_WORDS = {
    "but", "however", "though", "although", "yet", "except"
}


# =========================
# 2. Helper functions
# =========================

def safe_text(text) -> str:
    if pd.isna(text):
        return ""
    return str(text).strip()


def tokenize(text: str):
    return re.findall(r"\b[\w']+\b", text.lower())


def contains_any_word(tokens, vocab):
    return any(tok in vocab for tok in tokens)


def count_all_caps_words(text: str) -> int:
    words = re.findall(r"\b[A-Z]{3,}\b", text)
    return len(words)


def detect_sarcasm_rule_based(text: str):
    """
    Returns:
        score: float in [0, 1]
        label: str
        triggers: list[str]
    """
    text = safe_text(text)
    text_lower = text.lower()
    tokens = tokenize(text)

    score = 0
    triggers = []

    if not text:
        return 0.0, "not_sarcastic", []

    # -------------------------
    # Rule 1: explicit phrases
    # -------------------------
    for pattern in SARCASTIC_PHRASES:
        if re.search(pattern, text_lower):
            score += 3
            triggers.append(f"phrase:{pattern}")

    # --------------------------------------------------
    # Rule 2: positive wording + strong negative wording
    # --------------------------------------------------
    has_positive = contains_any_word(tokens, POSITIVE_WORDS)
    has_negative = contains_any_word(tokens, NEGATIVE_WORDS)

    if has_positive and has_negative:
        score += 2
        triggers.append("positive_negative_conflict")

    # ------------------------------------------------
    # Rule 3: contrast marker + positive then negative
    # Example: "great movie but boring ending"
    # ------------------------------------------------
    if any(c in tokens for c in CONTRAST_WORDS) and has_positive and has_negative:
        score += 2
        triggers.append("contrast_flip")

    # -----------------------------
    # Rule 4: rhetorical questions
    # -----------------------------
    rhetorical_patterns = [
        r"\breally\?\b",
        r"\bseriously\?\b",
        r"\bwho thought\b",
        r"\bwhat could possibly\b",
        r"\bisn't that just\b",
        r"\bhow convenient\b"
    ]
    for pattern in rhetorical_patterns:
        if re.search(pattern, text_lower):
            score += 2
            triggers.append(f"rhetorical:{pattern}")

    if text.count("?") >= 2:
        score += 1
        triggers.append("multiple_question_marks")

    # -----------------------------------
    # Rule 5: exaggerated punctuation
    # -----------------------------------
    if "!!!" in text or "???" in text or "!?" in text or "?!" in text:
        score += 1
        triggers.append("exaggerated_punctuation")

    # -----------------------------
    # Rule 6: excessive ALL CAPS
    # -----------------------------
    all_caps_count = count_all_caps_words(text)
    if all_caps_count >= 2:
        score += 1
        triggers.append("all_caps_emphasis")

    # --------------------------------------
    # Rule 7: scare quotes around positives
    # Example: this was a "great" movie
    # --------------------------------------
    quoted_positive_patterns = [
        r'"great"', r'"amazing"', r'"awesome"', r'"perfect"', r'"brilliant"',
        r"'great'", r"'amazing'", r"'awesome'", r"'perfect'", r"'brilliant'"
    ]
    for pattern in quoted_positive_patterns:
        if re.search(pattern, text_lower):
            score += 2
            triggers.append("scare_quotes_positive")
            break

    # --------------------------------------
    # Rule 8: short sarcastic interjections
    # --------------------------------------
    if re.search(r"\b(lol|lmao|rofl)\b", text_lower) and has_negative:
        score += 1
        triggers.append("mocking_laughter")

    # --------------------------------------
    # Convert score to label
    # --------------------------------------
    # You can tune these thresholds
    if score >= 5:
        label = "sarcastic"
    elif score >= 3:
        label = "possible_sarcasm"
    else:
        label = "not_sarcastic"

    # optional normalized score
    normalized_score = min(score / 8, 1.0)

    return normalized_score, label, triggers


In [ ]:
# =========================
# 3. Apply to CSV
# =========================

df = pl.read_excel(r'D:\Github\4021-Information-Retrieval\classification\evaluation\evaluation_dataset.xlsx').to_pandas()

results = df["content"].apply(detect_sarcasm_rule_based)

df["sarcasm_score"] = results.apply(lambda x: x[0])
df["sarcasm_label"] = results.apply(lambda x: x[1])
df["sarcasm_triggers"] = results.apply(lambda x: ", ".join(x[2]))

df.to_csv("rule_based_sarcasm.csv", index=False)